In [22]:
import numpy as np
import ot
import time

from tractot.proj_GMM import proj_GMM_along
from tractot.utils import random_MASs
from tractot.distances import distances_aniso_along

In [23]:
d=3
k1=5
nb_MAS=1
nb_MAP=5
_,_,_,w1,S1=random_MASs(nb_MAS,nb_MAP,k1,d)
w1,S1=w1[0],S1[0]
print(w1.shape,S1.shape)

(5, 5) (5, 5, 3, 3)


In [24]:
# Sanity check
k2,ksize=2,2
w2,S2=proj_GMM_along(k2,ksize,w1,S1)

print(w2.shape,S2.shape)

(5, 2) (5, 2, 3, 3)


In [25]:
# Flexible number of compartment

k2,ksize=np.array([2,3,2,2,3]),3
w2,S2=proj_GMM_along(k2,ksize,w1,S1)

print(w2.shape,S2.shape)
print(w2)

(5, 3) (5, 3, 3, 3)
[[0.69993749 0.03085883 0.        ]
 [0.34835996 0.19405875 0.21478305]
 [0.62452086 0.30642371 0.        ]
 [0.51587161 0.4056881  0.        ]
 [0.40044634 0.18547586 0.26664881]]


In [41]:
# Initialization
def sum_cost_MW(w1,S1,w2,S2):
    C=distances_aniso_along(S1,S2)
    cost=0
    for i in range(w1.shape[0]):
        cost+=ot.emd2(w[i],w2[i],C[i])
    return cost
    
n_seed=10
nb_MAS,nb_MAP,k,k2=1,10,20,3

res=np.zeros((n_seed,4))
res_time=np.zeros((n_seed,4))
for i in range(n_seed):
    print(i,end=' ')
    _,_,_,w,S=random_MASs(nb_MAS,nb_MAP,k1,d)
    w,S=w[0],S[0]

    t=time.time()
    w2,S2=proj_GMM_along(k2,k2,w,S,max_itr=50,eps=1e-9,init='random')
    res[i,0]=sum_cost_MW(w,S,w2,S2)
    res_time[i,0]=time.time()-t

    t=time.time()
    w2,S2=proj_GMM_along(k2,k2,w,S,max_itr=50,eps=1e-9,init='most weighted')
    res[i,1]=sum_cost_MW(w,S,w2,S2)
    res_time[i,1]=time.time()-t

    t=time.time()
    w2,S2=proj_GMM_along(k2,k2,w,S,max_itr=50,eps=1e-9,init='plusplus')
    res[i,2]=sum_cost_MW(w,S,w2,S2)
    res_time[i,2]=time.time()-t

    t=time.time()
    w2,S2=proj_GMM_along(k2,k2,w,S,max_itr=50,eps=1e-9,init='plusplus_weighted')
    res[i,3]=sum_cost_MW(w,S,w2,S2)
    res_time[i,3]=time.time()-t
print()

print(res.sum(0))
print(res_time.sum(0))

0 1 2 3 4 5 6 7 8 9 
[22.33412609 17.24225939 18.68714515 17.12856561]
[0.52695918 0.30479288 0.39558864 0.35237455]
